In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'telco-customer-churn' dataset.
Path to dataset files: /kaggle/input/telco-customer-churn


In [2]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, classification_report

# Construct the full path to the CSV file
file_name = 'WA_Fn-UseC_-Telco-Customer-Churn.csv' # Common file name for this dataset
data_path = os.path.join(path, file_name)

# Load the dataset
df = pd.read_csv(data_path)

# Display the first 5 rows of the DataFrame
print("First 5 rows of the dataset:")
display(df.head())

# Display information about the DataFrame to check data types and missing values
print("\nDataFrame Info:")
df.info()

First 5 rows of the dataset:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 no

In [3]:
# Drop 'customerID' column
df = df.drop('customerID', axis=1)

# Convert 'TotalCharges' to numeric, coercing errors to NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Handle missing values in 'TotalCharges' (impute with median or mean)
# For simplicity, let's fill with the median for now
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Convert 'Churn' target variable to numerical (Yes=1, No=0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Identify categorical and numerical features
categorical_features = df.select_dtypes(include='object').columns
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.drop('Churn')

print("Categorical Features:", categorical_features.tolist())
print("Numerical Features (excluding Churn):", numerical_features.tolist())

# Apply one-hot encoding to categorical features
df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=True)

print("\nDataFrame after encoding:")
display(df_encoded.head())

Categorical Features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Numerical Features (excluding Churn): ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

DataFrame after encoding:


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False


In [4]:
# Define features (X) and target (y)
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

# Initialize and train the Logistic Regression model
model = LogisticRegression(max_iter=1000, solver='liblinear', random_state=42) # Increased max_iter for convergence
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1] # Probabilities for the positive class (Churn=1)

# --- Evaluate the model ---

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\nAccuracy: {accuracy:.4f}")

# Classification Report (includes precision, recall, f1-score)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Mean Squared Error (MSE)
mse = mean_squared_error(y_test, y_pred)
print(f"\nMean Squared Error (MSE): {mse:.4f}")

# Mean Absolute Error (MAE)
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error (MAE): {mae:.4f}")

# R-squared
# For logistic regression, R-squared is not a standard metric as it is for linear regression.
# We can calculate it against the predicted classes, but it won't have the same interpretation.
r2 = r2_score(y_test, y_pred)
print(f"R-squared (against predicted classes): {r2:.4f}")

# Alternatively, calculate R-squared for predicted probabilities (less common but can be done)
r2_proba = r2_score(y_test, y_pred_proba)
print(f"R-squared (against predicted probabilities): {r2_proba:.4f}")

Training set size: 4930 samples
Test set size: 2113 samples

Accuracy: 0.8083

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.90      0.87      1552
           1       0.67      0.56      0.61       561

    accuracy                           0.81      2113
   macro avg       0.76      0.73      0.74      2113
weighted avg       0.80      0.81      0.80      2113


Mean Squared Error (MSE): 0.1917
Mean Absolute Error (MAE): 0.1917
R-squared (against predicted classes): 0.0171
R-squared (against predicted probabilities): 0.3013


In [5]:
import joblib

# Save the trained model
joblib.dump(model, 'logistic_regression_model.pkl')

# Save the list of feature names (columns) that the model was trained on
joblib.dump(X.columns.tolist(), 'model_features.pkl')

# Save the list of original categorical features (used for one-hot encoding)
categorical_cols = df.select_dtypes(include='object').columns.tolist()
joblib.dump(categorical_cols, 'categorical_features.pkl')

print("Model, feature list, and categorical features saved successfully.")

Model, feature list, and categorical features saved successfully.


In [6]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib
import numpy as np

# Load the trained model and feature list
model = joblib.load('logistic_regression_model.pkl')
model_features = joblib.load('model_features.pkl')
categorical_features = joblib.load('categorical_features.pkl') # Original categorical features

st.set_page_config(page_title="Telco Customer Churn Predictor", layout="centered")
st.title("Predict Telco Customer Churn")
st.write("Enter customer details to predict if they will churn or not.")

# --- Input Form ---
with st.form("churn_prediction_form"):
    st.header("Customer Demographics")
    gender = st.selectbox("Gender", ['Male', 'Female'])
    SeniorCitizen = st.selectbox("Senior Citizen", [0, 1], format_func=lambda x: 'Yes' if x==1 else 'No')
    Partner = st.selectbox("Partner", ['Yes', 'No'])
    Dependents = st.selectbox("Dependents", ['Yes', 'No'])

    st.header("Services")
    tenure = st.slider("Tenure (months)", min_value=1, max_value=72, value=12)
    PhoneService = st.selectbox("Phone Service", ['Yes', 'No'])
    MultipleLines = st.selectbox("Multiple Lines", ['No phone service', 'No', 'Yes'])
    InternetService = st.selectbox("Internet Service", ['DSL', 'Fiber optic', 'No'])
    OnlineSecurity = st.selectbox("Online Security", ['No internet service', 'No', 'Yes'])
    OnlineBackup = st.selectbox("Online Backup", ['No internet service', 'No', 'Yes'])
    DeviceProtection = st.selectbox("Device Protection", ['No internet service', 'No', 'Yes'])
    TechSupport = st.selectbox("Tech Support", ['No internet service', 'No', 'Yes'])
    StreamingTV = st.selectbox("Streaming TV", ['No internet service', 'No', 'Yes'])
    StreamingMovies = st.selectbox("Streaming Movies", ['No internet service', 'No', 'Yes'])

    st.header("Billing Information")
    Contract = st.selectbox("Contract Type", ['Month-to-month', 'One year', 'Two year'])
    PaperlessBilling = st.selectbox("Paperless Billing", ['Yes', 'No'])
    PaymentMethod = st.selectbox("Payment Method", ['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'])
    MonthlyCharges = st.slider("Monthly Charges", min_value=18.0, max_value=120.0, value=50.0, step=0.01)
    TotalCharges = st.number_input("Total Charges", min_value=0.0, value=200.0, step=0.01)

    submitted = st.form_submit_button("Predict Churn")

# --- Prediction Logic ---
if submitted:
    # Create a DataFrame from the input values
    input_data = pd.DataFrame({
        'gender': [gender],
        'SeniorCitizen': [SeniorCitizen],
        'Partner': [Partner],
        'Dependents': [Dependents],
        'tenure': [tenure],
        'PhoneService': [PhoneService],
        'MultipleLines': [MultipleLines],
        'InternetService': [InternetService],
        'OnlineSecurity': [OnlineSecurity],
        'OnlineBackup': [OnlineBackup],
        'DeviceProtection': [DeviceProtection],
        'TechSupport': [TechSupport],
        'StreamingTV': [StreamingTV],
        'StreamingMovies': [StreamingMovies],
        'Contract': [Contract],
        'PaperlessBilling': [PaperlessBilling],
        'PaymentMethod': [PaymentMethod],
        'MonthlyCharges': [MonthlyCharges],
        'TotalCharges': [TotalCharges]
    })

    # Convert 'TotalCharges' to numeric, handle missing (though inputs should be clean)
    input_data['TotalCharges'] = pd.to_numeric(input_data['TotalCharges'], errors='coerce')
    input_data['TotalCharges'] = input_data['TotalCharges'].fillna(input_data['TotalCharges'].median())

    # Apply one-hot encoding to categorical features from the original training data
    # Ensure consistent column order and handling of unseen categories
    input_encoded = pd.get_dummies(input_data, columns=categorical_features, drop_first=True)

    # Align columns with the training data (model_features)
    # Add missing columns with 0, and remove extra columns
    missing_cols = set(model_features) - set(input_encoded.columns)
    for c in missing_cols:
        input_encoded[c] = 0
    input_encoded = input_encoded[model_features] # Ensure the order of columns is the same

    # Make prediction
    prediction = model.predict(input_encoded)[0]
    prediction_proba = model.predict_proba(input_encoded)[0][1] # Probability of churn

    st.subheader("Prediction Result:")
    if prediction == 1:
        st.error(f"The customer is likely to CHURN! (Probability: {prediction_proba:.2%})")
        st.markdown("We recommend reaching out to this customer with retention offers.")
    else:
        st.success(f"The customer is likely to STAY. (Probability: {prediction_proba:.2%})")
        st.markdown("This customer seems satisfied. Continue providing excellent service!")

    st.markdown("--- Generative AI Analysis ---")
    if prediction == 1 and prediction_proba > 0.6:
        st.markdown("**Insights for churn:** High probability of churn often indicates dissatisfaction with pricing, service quality, or competitor offers. Consider targeted discounts or personalized service improvements.")
    elif prediction == 0 and prediction_proba < 0.4:
        st.markdown("**Insights for retention:** Low probability of churn suggests high customer satisfaction. Focus on loyalty programs or upselling relevant services.")
    else:
        st.markdown("**General insight:** Monitor customer engagement and satisfaction regularly to proactively address potential issues.")

Writing app.py


In [7]:
import ipywidgets as widgets
from IPython.display import display, HTML
import pandas as pd
import joblib
import numpy as np

# Load the trained model and feature list
model = joblib.load('logistic_regression_model.pkl')
model_features = joblib.load('model_features.pkl')
categorical_features = joblib.load('categorical_features.pkl')

print("Model and features loaded for in-notebook UI.")

Model and features loaded for in-notebook UI.


In [8]:
# Define widgets for each input feature

# Demographics
gender_widget = widgets.Dropdown(options=['Male', 'Female'], description='Gender:')
senior_citizen_widget = widgets.Checkbox(value=False, description='Senior Citizen:')
partner_widget = widgets.Checkbox(value=False, description='Partner:')
dependents_widget = widgets.Checkbox(value=False, description='Dependents:')

# Services
tenure_widget = widgets.IntSlider(min=1, max=72, step=1, value=12, description='Tenure (months):')
phone_service_widget = widgets.Checkbox(value=False, description='Phone Service:')
multiple_lines_widget = widgets.Dropdown(options=['No phone service', 'No', 'Yes'], description='Multiple Lines:')
internet_service_widget = widgets.Dropdown(options=['DSL', 'Fiber optic', 'No'], description='Internet Service:')
online_security_widget = widgets.Dropdown(options=['No internet service', 'No', 'Yes'], description='Online Security:')
online_backup_widget = widgets.Dropdown(options=['No internet service', 'No', 'Yes'], description='Online Backup:')
device_protection_widget = widgets.Dropdown(options=['No internet service', 'No', 'Yes'], description='Device Protection:')
tech_support_widget = widgets.Dropdown(options=['No internet service', 'No', 'Yes'], description='Tech Support:')
streaming_tv_widget = widgets.Dropdown(options=['No internet service', 'No', 'Yes'], description='Streaming TV:')
streaming_movies_widget = widgets.Dropdown(options=['No internet service', 'No', 'Yes'], description='Streaming Movies:')

# Billing Information
contract_widget = widgets.Dropdown(options=['Month-to-month', 'One year', 'Two year'], description='Contract Type:')
paperless_billing_widget = widgets.Checkbox(value=False, description='Paperless Billing:')
payment_method_widget = widgets.Dropdown(options=['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'], description='Payment Method:')
monthly_charges_widget = widgets.FloatSlider(min=18.0, max=120.0, step=0.01, value=50.0, description='Monthly Charges:')
total_charges_widget = widgets.FloatText(value=200.0, description='Total Charges:')

# Prediction button and output area
predict_button = widgets.Button(description='Predict Churn', button_style='success')
output_area = widgets.Output()

# Arrange widgets in a VBox
form_layout = widgets.VBox([
    widgets.HTML('<h3>Customer Demographics</h3>'),
    gender_widget,
    senior_citizen_widget,
    partner_widget,
    dependents_widget,
    widgets.HTML('<h3>Services</h3>'),
    tenure_widget,
    phone_service_widget,
    multiple_lines_widget,
    internet_service_widget,
    online_security_widget,
    online_backup_widget,
    device_protection_widget,
    tech_support_widget,
    streaming_tv_widget,
    streaming_movies_widget,
    widgets.HTML('<h3>Billing Information</h3>'),
    contract_widget,
    paperless_billing_widget,
    payment_method_widget,
    monthly_charges_widget,
    total_charges_widget,
    predict_button,
    output_area
])

display(form_layout)

In [9]:
def on_predict_button_clicked(b):
    with output_area:
        output_area.clear_output()

        # Collect input data
        input_data = pd.DataFrame({
            'gender': [gender_widget.value],
            'SeniorCitizen': [1 if senior_citizen_widget.value else 0],
            'Partner': ['Yes' if partner_widget.value else 'No'],
            'Dependents': ['Yes' if dependents_widget.value else 'No'],
            'tenure': [tenure_widget.value],
            'PhoneService': ['Yes' if phone_service_widget.value else 'No'],
            'MultipleLines': [multiple_lines_widget.value],
            'InternetService': [internet_service_widget.value],
            'OnlineSecurity': [online_security_widget.value],
            'OnlineBackup': [online_backup_widget.value],
            'DeviceProtection': [device_protection_widget.value],
            'TechSupport': [tech_support_widget.value],
            'StreamingTV': [streaming_tv_widget.value],
            'StreamingMovies': [streaming_movies_widget.value],
            'Contract': [contract_widget.value],
            'PaperlessBilling': ['Yes' if paperless_billing_widget.value else 'No'],
            'PaymentMethod': [payment_method_widget.value],
            'MonthlyCharges': [monthly_charges_widget.value],
            'TotalCharges': [total_charges_widget.value]
        })

        # Preprocess input data to match model training format
        input_data['TotalCharges'] = pd.to_numeric(input_data['TotalCharges'], errors='coerce')
        input_data['TotalCharges'] = input_data['TotalCharges'].fillna(input_data['TotalCharges'].median()) # Should ideally not have NaNs from UI

        # Apply one-hot encoding using the original categorical features
        input_encoded = pd.get_dummies(input_data, columns=categorical_features, drop_first=True)

        # Align columns with the training data (model_features)
        # Add missing columns with 0, and remove extra columns
        missing_cols = set(model_features) - set(input_encoded.columns)
        for c in missing_cols:
            input_encoded[c] = 0
        input_encoded = input_encoded[model_features] # Ensure the order of columns is the same

        # Make prediction
        prediction = model.predict(input_encoded)[0]
        prediction_proba = model.predict_proba(input_encoded)[0][1] # Probability of churn

        # Display result
        if prediction == 1:
            display(HTML(f"<p style='color:red; font-size:1.2em;'><b>Prediction: Customer is likely to CHURN! (Probability: {prediction_proba:.2%})</b></p>"))
            display(HTML("<p><i>Recommendation: Consider targeted retention offers or service improvements.</i></p>"))
        else:
            display(HTML(f"<p style='color:green; font-size:1.2em;'><b>Prediction: Customer is likely to STAY. (Probability: {prediction_proba:.2%})</b></p>"))
            display(HTML("<p><i>Recommendation: Continue providing excellent service!</i></p>"))

        # Add AI analysis based on probability
        display(HTML("<hr><h4>Generative AI Analysis:</h4>"))
        if prediction == 1 and prediction_proba > 0.6:
            display(HTML("<p>High probability of churn often indicates dissatisfaction with pricing, service quality, or competitor offers. Consider targeted discounts or personalized service improvements.</p>"))
        elif prediction == 0 and prediction_proba < 0.4:
            display(HTML("<p>Low probability of churn suggests high customer satisfaction. Focus on loyalty programs or upselling relevant services.</p>"))
        else:
            display(HTML("<p>Monitor customer engagement and satisfaction regularly to proactively address potential issues.</p>"))

# Attach the event handler to the button
predict_button.on_click(on_predict_button_clicked)